In [0]:
!pip install requests

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


Passo 1 · Extrair: pedir os feriados à API

Célula 1

In [0]:
import requests                # conversa com a API (HTTP)

url = "https://brasilapi.com.br/api/feriados/v1/2026"
resposta = requests.get(url)

feriados = resposta.json()     # o JSON vira uma lista do Python: um dicionário por feriado
print(len(feriados))


14


Passo 2 · Pousar: o JSON numa pasta nova do raw
Cada execução grava um arquivo novo, com a data e a hora no nome. O raw guarda o histórico de tudo o que chegou: dá para saber o que a API respondia em cada momento.

Célula 2

In [0]:
import json, os
from datetime import datetime

pasta = "/Volumes/lakehouse_vertice/bronze/raw/feriados"                 # a pasta nova precisa existir

agora = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")     # ex.: 2026-10-20_19-40-00 (horário UTC)
with open(f"{pasta}/feriados_{agora}.json", "w") as arquivo:
    json.dump(feriados, arquivo, ensure_ascii=False)

sorted(os.listdir(pasta))                                # um arquivo por execução

['feriados_2026-10-21_19-43-38.json',
 'feriados_2026-10-21_20-55-51.json',
 'feriados_2026-10-21_20-56-16.json']

Passo 3 · Carregar: o raw vira a bronze
O mesmo read_files da noite 1, lendo todos os arquivos da pasta. Duas colunas a mais contam de onde e de quando veio cada linha.

Célula 3

In [0]:
%sql
CREATE OR REPLACE TABLE lakehouse_vertice.bronze.feriados
COMMENT 'Feriados nacionais de 2026, da BrasilAPI. Uma captura por execução do job.'
AS SELECT date AS data, name AS feriado, type AS tipo, weekday AS dia_da_semana,
          _metadata.file_name              AS arquivo,
          _metadata.file_modification_time AS coletado_em
FROM read_files('/Volumes/lakehouse_vertice/bronze/raw/feriados/', format => 'json', multiLine => true)

num_affected_rows,num_inserted_rows
